In [9]:
%pip install ollama tqdm

Note: you may need to restart the kernel to use updated packages.


In [16]:
import asyncio
import json
import os
import time
from datetime import datetime

from ollama import AsyncClient
from tqdm import tqdm

In [17]:
# Define the prisoner's dilemma prompt separately for clarity
PRISONER_DILEMMA_PROMPT = """
You are facing a prisoner's dilemma:
- If both you and your partner stay silent, you each receive a short sentence.
- If you implicate your partner and they stay silent, you go free, and they receive a long sentence.
- If you stay silent while they implicate you, you receive a long sentence, and they go free.
- If you both implicate each other, you both receive a medium sentence.

What would you do? Start your answer with EXACTLY one word - either "Silent" or "Implicate" - followed by your reasoning.
"""

In [18]:
async def get_ollama_response_clean_async(client: AsyncClient, iteration: int):
    """Get response from Ollama asynchronously."""
    try:
        start_time = time.perf_counter()
        response = await client.generate(
            model="llama3.2:latest",
            prompt=PRISONER_DILEMMA_PROMPT,
            options={"temperature": 0.7},
        )
        end_time = time.perf_counter()
        response_text = response.response if hasattr(response, "response") else response["response"]
        return {
            "response_text": response_text,
            "iteration": iteration,
            "response_time": end_time - start_time,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }
    except Exception as e:
        return {
            "iteration": iteration,
            "error": str(e),
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }

In [19]:
async def run_simple_experiment_async(iterations=50, batch_size=5):
    """Run the experiment asynchronously: 50 runs in batches of 5 (10 batches)."""
    output_dir = "experiment_results"
    os.makedirs(output_dir, exist_ok=True)
    output_prefix = f"{output_dir}/prisoner_dilemma_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    full_results = []
    num_batches = (iterations + batch_size - 1) // batch_size

    print(f"Running {iterations} iterations in {num_batches} batches of {batch_size} (async, llama3.2:latest)...")
    async with AsyncClient() as client:
        for batch_idx in tqdm(range(num_batches), desc="Batches"):
            batch_start = batch_idx * batch_size + 1
            batch_end = min(batch_start + batch_size, iterations + 1)
            batch_iters = list(range(batch_start, batch_end))
            tasks = [get_ollama_response_clean_async(client, i) for i in batch_iters]
            batch_results = await asyncio.gather(*tasks, return_exceptions=False)
            full_results.extend(batch_results)
            if batch_idx < num_batches - 1:
                await asyncio.sleep(0.5)

    full_results.sort(key=lambda r: r.get("iteration", 0))
    valid_results = [r for r in full_results if "error" not in r]

    full_output_file = f"{output_prefix}_full_results.json"
    with open(full_output_file, "w") as f:
        json.dump(full_results, f, indent=2)

    print(f"\nExperiment completed. {len(valid_results)}/{len(full_results)} successful.")
    print(f"Full results saved to: {full_output_file}")
    if full_results:
        print("\nSample raw output:")
        print(json.dumps(full_results[0], indent=2))

    return full_results, valid_results, output_prefix

# Run 50 iterations in batches of 5 (fully async)
# Use await in Jupyter — it already has a running event loop; asyncio.run() would fail.
full_results, valid_results, output_prefix = await run_simple_experiment_async(
    iterations=50, batch_size=5
)

RuntimeError: asyncio.run() cannot be called from a running event loop